In [7]:
from openai import OpenAI
from website_scraper import scrape_website
import json

In [12]:
ollama = OpenAI(base_url="http://localhost:11434/v1" , api_key= "ollama")
MODEL = "llama3.1:8b"

In [2]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    webb = scrape_website(url)
    links = webb.links
    user_prompt += "\n".join(links)
    return user_prompt

In [17]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling model:{MODEL}")
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links
    

In [16]:
webb = scrape_website("http://bbc.com")
webb.links

['http://bbc.com/news/articles/cwydexp39ddo',
 'http://bbc.com/travel/adventures',
 'https://www.bbc.co.uk/contact',
 'http://bbc.com/audio',
 'http://bbc.com/news/articles/cgrlqxlwerqo',
 'https://www.bbc.com/news/us-canada',
 'http://bbc.com/live',
 'http://bbc.com/sport/football/articles/c5ywz3x4r2eo',
 'http://bbc.com/future-planet/solutions',
 'http://bbc.com/audio/categories',
 'http://bbc.com/undefined',
 'http://bbc.com/news/world/europe',
 'https://shop.bbc.com/',
 'https://www.bbc.com/editorialguidelines/guidance/links-and-feeds',
 'http://bbc.com/culture/music',
 'http://bbc.com/reel/video/p08kz96l/watch',
 'http://bbc.com/news/scotland/scotland_politics',
 'http://bbc.com/news/articles/cy0124xel35o',
 'http://bbc.com/reel/video/p0515qql/watch',
 'http://bbc.com/travel/article/20260316-the-turkish-city-built-on-green-gold',
 'https://www.bbc.com/ukrainian',
 'https://www.bbc.com/burmese',
 'http://bbc.com/news/articles/crr185nx0n9o',
 'http://bbc.com/audio/play/w3ct8svv',
 '

In [14]:
select_relevant_links("https://bbc.com")

{'links': [{'type': 'about page', 'url': 'https://www.bbc.co.uk/contact'},
  {'type': 'news page', 'url': 'https://bbc.com/news/videos/cq6jpp2e5d6o'},
  {'type': 'jobs page', 'url': 'https://www.bbc.co.uk/aboutthebbc'},
  {'type': 'careers page', 'url': '$blank'},
  {'type': 'company page', 'url': 'https://bbc.com/'},
  {'type': 'mission/vision page', 'url': '$blank'}]}

In [18]:
def fetch_page_and_relevant_links(url):
    website = scrape_website(url)
    contents = website.text
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        w = scrape_website(link["url"])
        result += f"\n\n### Link: {link['type']}\n"
        result += w.text
    return result

In [19]:
print(fetch_page_and_relevant_links("https://bbc.com"))

Selecting relevant links for https://bbc.com by calling model:llama3.1:8b
Found 7 relevant links
## Landing Page:

BBC Home - Breaking News, World News, US News, Sports, Business, Innovation, Climate, Culture, Travel, Video & Audio
Skip to content
British Broadcasting Corporation
Register
Sign In
Home
News
Sport
Business
Technology
Health
Culture
Arts
Travel
Earth
Audio
Video
Live
Home
News
US & Canada
UK
UK Politics
England
N. Ireland
N. Ireland Politics
Scotland
Scotland Politics
Wales
Wales Politics
Africa
Asia
China
India
Australia
Europe
Latin America
Middle East
In Pictures
BBC InDepth
BBC Verify
Sport
Business
World of Business
Technology of Business
NYSE Opening Bell
Technology
Artificial Intelligence
Intelligence Revolution
AI v the Mind
Health
Culture
Film & TV
Music
Art & Design
Style
Books
Entertainment News
Arts
Arts in Motion
Travel
Destinations
Africa
Antarctica
Asia
Australia and Pacific
Caribbean & Bermuda
Central America
Europe
Middle East
North America
South America


In [20]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
from IPython.display import Markdown

def create_brochure(company_name, url):
    # stream = ollama.chat.completions.create(
    response = ollama.chat.completions.create(
    
        model = MODEL,
        messages=[
            {"role":"system" , "content":brochure_system_prompt},
            {"role":"user" , "content": get_brochure_user_prompt(company_name,url)}
        ]
        # stream = True
    )
    # response = ""
    # display_handle = display(Markdown(""), display_id=True)
    # for chunk in stream:
    #     response += chunk.choices[0].delta.content or ''
    #     update_display(Markdown(response), display_id=display_handle.display_id)
    result = response.choices[0].message.content
    display(Markdown(result))


In [26]:
create_brochure("BBC","https://bbc.com")

Selecting relevant links for https://bbc.com by calling model:llama3.1:8b
Found 4 relevant links


**Welcome to the British Broadcasting Corporation (BBC)**

The BBC is a world-renowned public service broadcaster that provides high-quality news, information, and entertainment to audiences around the globe. Our mission is to inform, educate, and entertain, while promoting diversity and inclusivity in our content.

**Key Highlights:**

* The BBC offers a wide range of programming in various genres, including news, sport, business, technology, health, culture, arts, travel, and more.
* We have a global reach, with a presence in over 200 countries and territories through our international partnerships and collaborations.
* Our content is available in multiple languages, making us accessible to diverse audiences worldwide.

**Company Culture:**

* The BBC values diversity, equity, and inclusion in all aspects of our operations. We strive to create a work environment that is welcoming, inclusive, and fair for everyone.
* We are committed to promoting innovation, creativity, and excellence in all areas of our business.

**Our Customers:**

* Our audience spans the globe, with over 456 million people accessing our services every week.
* We cater to diverse interests and preferences through a wide range of programming, including news, sport, music, film, and more.

**Careers at the BBC:**

* The BBC offers exciting job opportunities across various fields, including journalism, technology, marketing, and creative industries.
* We value talent, experience, and passion for storytelling, innovation, and public service broadcasting.

**Join Our Community:**

* Become a part of our global network of talented individuals who are passionate about sharing stories, ideas, and perspectives with the world.
* Engage with us on social media to stay up-to-date with our latest news, behind-the-scenes insights, and more.